In [0]:
# bronze dlt - auto loader reads .json files from the landing zone
import dlt
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

CATALOG = spark.conf.get("source_catalog", "dev")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/landing_zone/liquor_sales/"

LIQUOR_SCHEMA = StructType([
    StructField("invoice_line_no", StringType(), True),
    StructField("date", StringType(), True),
    StructField("store", StringType(), True),
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("county", StringType(), True),
    StructField("category", StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("vendor_no", StringType(), True),
    StructField("vendor_name", StringType(), True),
    StructField("itemno", StringType(), True),
    StructField("im_desc", StringType(), True),
    StructField("pack", StringType(), True),
    StructField("bottle_volume_ml", StringType(), True),
    StructField("state_bottle_cost", StringType(), True),
    StructField("state_bottle_retail", StringType(), True),
    StructField("sale_bottles", StringType(), True),
    StructField("sale_dollars", StringType(), True),
    StructField("sale_liters", StringType(), True),
    StructField("sale_gallons", StringType(), True),
])

@dlt.table(
    name="raw_liquor_sales",
    comment="Bronze - raw datasets fetched from api, append-only via auto loader",
    table_properties={"quality": "bronze"}
)
def raw_liquor_sales():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaEvolutionMode", "none")
        .schema(LIQUOR_SCHEMA)
        .load(VOLUME_PATH)
        .withColumn("_ingestion_ts", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )